# AP01 – Setup

## Libraries und Arbeitsverzeichnis

- `pandas` liest Excel-Dateien ein und verarbeitet Daten als DataFrames
- `create_engine()` richtet den Zugriff auf PostgreSQL ein
- `text()` bereitet SQL-Anweisungen für die Ausführung über eine SQLAlchemy-Verbindung vor
- `Path.cwd()` zeigt den aktuellen Arbeitsordner an, von dem relative Dateipfade ausgehen
- Docker stellt in deinem Codespace den PostgreSQL-Datenbankserver bereit
Voraussetzung: PostgreSQL läuft und der passende Python-Kernel ist ausgewählt

In [1]:
#NICHTS ANPASSEN
import pandas as pd                                                             #pandas als pd laden für Excel-Dateien und DataFrames
from sqlalchemy import create_engine, text                                      #Werkzeuge für Datenbankzugriff und SQL-Anweisungen laden
from pathlib import Path                                                        #Werkzeug für Datei- und Ordnerpfade laden

print('Arbeitsverzeichnis', Path.cwd())                                         #Aktuellen Arbeitsordner anzeigen als Ausgangspunkt für relative Dateipfade

Arbeitsverzeichnis /workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01/AP01 Mock Exam


## Datenbankverbindung erstellen und testen

Die Verbindungsdaten stammen aus der mitgelieferten `docker-compose.yml`

- `user` und `password` sind die Zugangsdaten für PostgreSQL
- `host` ist die Adresse des Datenbankservers, hier `localhost`
- `port` ist der PostgreSQL-Port, hier `5432`
- `database` ist der Datenbankname, hier `postgres`
- `db_connection_url` setzt diese Angaben zu einer Verbindungsadresse zusammen
- `engine` verwaltet den Datenbankzugriff
- Erwartete Ausgabe: `Verbundene Datenbank postgres`
Ein erfolgreicher Verbindungstest bestätigt den Zugriff auf PostgreSQL, die benötigten Tabellen müssen anschliessend geladen werden, falls sie noch fehlen

In [2]:
#ANPASSEN user, password, host, port und database nur falls die Prüfungsumgebung andere Verbindungsdaten vorgibt
#REST BLEIBT GLEICH Verbindungsadresse, Engine und Verbindungstest sind unabhängig vom Datensatz
user = 'pgadmin'                                                                #PostgreSQL-Benutzer festlegen
password = 'geheim'                                                             #Passwort des Benutzers festlegen
host = 'localhost'                                                              #Adresse des PostgreSQL-Servers festlegen
port = '5432'                                                                   #Port des PostgreSQL-Servers festlegen
database = 'postgres'                                                           #Gewünschte Datenbank festlegen

db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'  #Verbindungsadresse aus den Zugangsdaten zusammensetzen

engine = create_engine(db_connection_url)                                       #Engine für den Datenbankzugriff erstellen

with engine.connect() as connection:                                            #Verbindung öffnen und nach dem eingerückten Block automatisch schliessen
    result = connection.execute(text('SELECT current_database()'))              #Namen der verbundenen Datenbank abfragen
    print('Verbundene Datenbank', result.scalar())                              #Einzelnen Ergebniswert auslesen und anzeigen

engine.dispose()                                                                #Vom Verbindungspool gehaltene Verbindungen freigeben

Verbundene Datenbank postgres


## Excel einlesen und Tabellen nach PostgreSQL laden

- `pd.read_excel()` liest jedes Excel-Tabellenblatt als DataFrame in Python ein
- `.to_sql()` schreibt den jeweiligen DataFrame als Tabelle nach PostgreSQL
- `sheet_name` bestimmt das einzulesende Excel-Tabellenblatt
- `.str.lower()` wandelt die Spaltennamen in Kleinbuchstaben um
- Der erste Parameter von `.to_sql()` bestimmt den SQL-Tabellennamen
- `if_exists='replace'` löscht eine vorhandene Tabelle und erstellt sie aus dem DataFrame neu
- `index=False` verhindert, dass der pandas-Zeilenindex als zusätzliche Spalte gespeichert wird
- ANPASSEN: Dateipfad, Sheet-Namen, DataFrame-Namen und SQL-Tabellennamen an den Prüfungsdatensatz anpassen
- Anderes Dateiformat: `pd.read_excel(path, sheet_name='…')` ersetzen durch `pd.read_csv(path)` für CSV, `pd.read_csv(path, sep='\t')` für tabulatorgetrennte TXT/TSV, `pd.read_csv(path, sep=';')` für semikolongetrennte Dateien, `pd.read_json(path)` für JSON oder `pd.read_parquet(path)` für Parquet. `sheet_name` entfällt und der anschliessende `.to_sql()`-Schritt bleibt gleich
- Den Import nur ausführen, wenn die Tabellen geladen oder bewusst neu erstellt werden sollen

In [3]:
#ANPASSEN path und sheet_name gemäss Prüfungsdatei und ihren Tabellenblättern sowie je Tabellenblatt die passenden Zeilen ergänzen oder entfernen
#ANPASSEN DataFrame-Namen durchgehend einheitlich verwenden und SQL-Tabellennamen in to_sql gemäss Aufgabenstellung festlegen
path = 'bike_shop_data.xlsx'                                                    #ANPASSEN Pfad zur Excel-Datei relativ zum angezeigten Arbeitsordner

df_bicycles = pd.read_excel(path, sheet_name='Bicycles')                        #Fahrraddaten aus dem Sheet Bicycles einlesen
df_customers = pd.read_excel(path, sheet_name='Customers')                      #Kundendaten aus dem Sheet Customers einlesen
df_sales = pd.read_excel(path, sheet_name='Sales')                              #Verkäufe aus dem Sheet Sales einlesen
df_salespersons = pd.read_excel(path, sheet_name='SalesPersons')                #Verkäuferdaten aus dem Sheet SalesPersons einlesen
df_mechanics = pd.read_excel(path, sheet_name='Mechanics')                      #Mechanikerdaten aus dem Sheet Mechanics einlesen
df_servicerecords = pd.read_excel(path, sheet_name='ServiceRecords')            #Servicedaten aus dem Sheet ServiceRecords einlesen

df_bicycles.columns = df_bicycles.columns.str.lower()                           #Spaltennamen der Fahrraddaten kleinschreiben
df_customers.columns = df_customers.columns.str.lower()                         #Spaltennamen der Kundendaten kleinschreiben
df_sales.columns = df_sales.columns.str.lower()                                 #Spaltennamen der Verkäufe kleinschreiben
df_salespersons.columns = df_salespersons.columns.str.lower()                   #Spaltennamen der Verkäuferdaten kleinschreiben
df_mechanics.columns = df_mechanics.columns.str.lower()                         #Spaltennamen der Mechanikerdaten kleinschreiben
df_servicerecords.columns = df_servicerecords.columns.str.lower()               #Spaltennamen der Servicedaten kleinschreiben

engine = create_engine(db_connection_url)                                       #Engine mit der zuvor erstellten Verbindungsadresse erstellen

df_bicycles.to_sql('bicycles', engine, if_exists='replace', index=False)        #Fahrraddaten als SQL-Tabelle bicycles speichern und vorhandene Tabelle ersetzen
df_customers.to_sql('customers', engine, if_exists='replace', index=False)      #Kundendaten als SQL-Tabelle customers speichern und vorhandene Tabelle ersetzen
df_sales.to_sql('sales', engine, if_exists='replace', index=False)              #Verkäufe als SQL-Tabelle sales speichern und vorhandene Tabelle ersetzen
df_salespersons.to_sql('salespersons', engine, if_exists='replace', index=False)      #Verkäuferdaten als SQL-Tabelle salespersons speichern und vorhandene Tabelle ersetzen
df_mechanics.to_sql('mechanics', engine, if_exists='replace', index=False)            #Mechanikerdaten als SQL-Tabelle mechanics speichern und vorhandene Tabelle ersetzen
df_servicerecords.to_sql('servicerecords', engine, if_exists='replace', index=False)  #Servicedaten als SQL-Tabelle servicerecords speichern und vorhandene Tabelle ersetzen

engine.dispose()                                                                #Vom Verbindungspool gehaltene Verbindungen freigeben

In [5]:
#Alternativ CSV

#path = 'bicycles.csv'                                                          #ANPASSEN Pfad zur CSV-Datei
#df_bicycles = pd.read_csv(path, sep=',')                                       #CSV als DataFrame einlesen und bei Bedarf Trennzeichen auf Semikolon ändern
#df_bicycles.columns = df_bicycles.columns.str.lower()                          #Spaltennamen in Kleinbuchstaben umwandeln
#engine = create_engine(db_connection_url)                                      #Engine mit der zuvor erstellten Verbindungsadresse erstellen
#df_bicycles.to_sql('bicycles', engine, if_exists='replace', index=False)       #Als SQL-Tabelle bicycles speichern und vorhandene Tabelle ersetzen
#engine.dispose()                                                               #Vom Verbindungspool gehaltene Verbindungen freigeben

## PostgreSQL Data Types
- Bsp.: `CAST(spalte AS datentyp)`, bspw. `CAST(year AS TEXT)`

| Datentyp zum Einsetzen | Bedeutung |
|---|---|
| `SMALLINT` | Kleine Ganzzahl von −32768 bis 32767 |
| `INTEGER` oder `INT` | Ganzzahl mit ca. ±2.1 Milliarden |
| `BIGINT` | Grosse Ganzzahl mit ca. ±9.2 Trillionen |
| `NUMERIC` oder `DECIMAL` | Exakte Dezimalzahl, geeignet für Geldbeträge und `ROUND(..., 2)` |
| `NUMERIC(10,2)` | Dezimalzahl mit maximal zehn Stellen insgesamt, davon zwei Nachkommastellen |
| `REAL` | Näherungsweise Kommazahl mit ca. sechs signifikanten Stellen |
| `DOUBLE PRECISION` | Näherungsweise Kommazahl mit ca. 15 signifikanten Stellen |
| `TEXT` | Text ohne festgelegte maximale Länge |
| `VARCHAR(100)` | Text mit maximal 100 Zeichen |
| `CHAR(10)` | Text mit fester Länge, kürzere Werte werden mit Leerzeichen aufgefüllt |
| `BOOLEAN` | Wahrheitswert `TRUE` oder `FALSE` |
| `DATE` | Datum ohne Uhrzeit |
| `TIME` | Uhrzeit ohne Datum und Zeitzone |
| `TIMETZ` | Uhrzeit mit Zeitzonenoffset, ohne Datum |
| `TIMESTAMP` | Datum und Uhrzeit ohne Zeitzone |
| `TIMESTAMPTZ` | Eindeutiger Zeitpunkt, Ausgabe gemäss Zeitzone der Sitzung |
| `INTERVAL` | Zeitdauer, bspw. zwei Tage |
| `UUID` | Universelle eindeutige Kennung |
| `JSON` | JSON-Daten als Textrepräsentation |
| `JSONB` | JSON-Daten in aufbereiteter binärer Form |
| `BYTEA` | Binärdaten, bspw. Dateiinhalte |
| `INTEGER[]` oder `TEXT[]` | Array mit mehreren Werten desselben Datentyps |

- Die Umwandlung muss möglich sein. `'2023'` lässt sich in `INTEGER` umwandeln, `'Bianchi'` nicht
- `CAST` verändert nur den Wert im Abfrageergebnis, nicht den gespeicherten Spaltentyp
- Dies sind die gängigen Datentypen, PostgreSQL bietet zusätzlich Spezialtypen, bspw. für Geometrie, Netzadressen und Wertebereiche

## PostgreSQL Calculations

| Berechnung | Ausdruck |
|---|---|
| Addieren | `price + 100` |
| Subtrahieren | `price - 100` |
| Multiplizieren | `price * 2` |
| Dividieren | `price / 2.0` |
| 10% Rabatt | `price * 0.90` |
| 5% Aufschlag | `price * 1.05` |
| 8.1% Mehrwertsteuer hinzufügen | `price * 1.081` |
| Anteil in Prozent | `teil * 100.0 / gesamt` |
| Auf zwei Dezimalstellen runden | `ROUND(CAST(ausdruck AS NUMERIC), 2)` |
| Potenz | `POWER(price, 2)` oder `price ^ 2` |
| Quadratwurzel | `SQRT(price)` |

- Im SELECT bspw. `price * 0.90 AS discounted_price`, gespeicherte Werte bleiben unverändert
- Punktrechnung vor Strichrechnung, Klammern steuern die Reihenfolge, bspw. `(price + 100) * 0.90`
- Ganzzahl durch Ganzzahl schneidet Nachkommastellen ab. `5 / 2` ergibt 2, `5 / 2.0` ergibt 2.5
- Division durch 0 vermeiden. `teil / NULLIF(gesamt, 0)` liefert bei Nenner 0 stattdessen NULL
- Berechnungen mit NULL ergeben bei diesen Rechenoperatoren NULL

## Troubleshoot – PostgreSQL-Container starten oder neu erstellen
- Die folgenden Befehle im Codespace-Terminal ausführen, der Container heisst `postgres_container`
- Bei technischen Problemen während der Prüfung umgehend die Prüfungsaufsicht informieren

### Containerstatus prüfen
```bash
docker ps -a                            #Alle Container inkl. gestoppter Container anzeigen
```

### Vorhandenen Container starten
```bash
docker start postgres_container         #Gestoppten PostgreSQL-Container starten
```

### Container neu starten
```bash
docker restart postgres_container       #PostgreSQL-Container stoppen und erneut starten
```

### Container neu erstellen
- Nur verwenden, wenn Starten oder Neustarten nicht hilft
- Im Ordner mit der bestehenden `docker-compose.yml` ausführen, `db` ist der Service-Name darin
```bash
docker compose up -d --force-recreate db #Datenbank-Container neu erstellen und im Hintergrund starten
```

### Alternative – Container gezielt entfernen und neu erstellen
- Alternative zum vorherigen Befehl, nicht zusätzlich erforderlich
- Ebenfalls im Ordner mit der bestehenden `docker-compose.yml` ausführen
```bash
docker rm -f postgres_container         #PostgreSQL-Container zwangsweise stoppen und entfernen
docker compose up -d db                 #Fehlenden Datenbank-Container neu erstellen und im Hintergrund starten
```

- Gemäss deiner Compose-Datei liegen die Daten im Volume `db_data`, das durch diese Befehle nicht gelöscht wird
- Die bestehende Compose-Datei unverändert verwenden, damit dasselbe Volume eingebunden wird
- Anschliessend kurz warten und den Verbindungstest im Notebook erneut ausführen
- Fehlende Tabellen bei Bedarf aus Excel laden
- Diese Befehle verwalten den Container, Datenbankabfragen ausschliesslich über Python im Notebook ausführen

# AP01 - SELECT

## Rules

### Spalten und Zeilen auswählen
- `SELECT` bestimmt die Ausgabespalten, `FROM` die verwendete Tabelle
- `SELECT *` wählt alle Spalten aus, ohne `WHERE` werden alle Zeilen berücksichtigt
- Spalten mit Kommas trennen, kein Komma direkt vor `FROM`
- `AS` benennt eine Ergebnisspalte um, die Originalspalte bleibt unverändert
- SELECT-Abfragen verändern die gespeicherten Daten nicht

### DISTINCT
- `DISTINCT` entfernt doppelte Ergebniszeilen
- Bei mehreren ausgewählten Spalten zählt die gesamte Kombination, `DISTINCT make, model` liefert eindeutige Kombinationen aus Marke und Modell
- Eine zusätzliche eindeutige ID verhindert meist das Entfernen von Duplikaten, weil jede Ergebniszeile dadurch unterschiedlich ist

### Werte und Vergleiche
- Text und Datumswerte in einfache Anführungszeichen setzen, Zahlen ohne Anführungszeichen
- Doppelte Anführungszeichen kennzeichnen Bezeichner, bspw. `"Listenpreis"`
- PostgreSQL wandelt Bezeichner ohne doppelte Anführungszeichen in Kleinbuchstaben um
- `=` gleich, `<>` oder `!=` ungleich, `<` kleiner, `>` grösser, `<=` und `>=` inkl. Gleichheit

### Bedingungen kombinieren
- `AND` verlangt alle Bedingungen, `OR` mind. eine
- `NOT` kehrt eine Bedingung um
- `AND` bindet stärker als `OR`, gewünschte Kombination einklammern
- Bspw. `(make = 'Bianchi' OR make = 'Trek') AND price > 3000`

### Bereiche und Wertelisten
- `BETWEEN 2000 AND 4000` schliesst beide Grenzen ein
- `NOT BETWEEN 2000 AND 4000` wählt Werte unter 2000 oder über 4000 aus
- `IN (...)` prüft auf einzelne erlaubte Werte, nicht auf einen zusammenhängenden Bereich
- `NOT IN (...)` schliesst die aufgeführten Werte aus
- Achtung bei `NOT IN` mit NULL in der Liste oder Unterabfrage. Nicht übereinstimmende Werte werden dann ebenfalls nicht ausgewählt

### Textmuster
- `LIKE` berücksichtigt Gross- und Kleinschreibung, `ILIKE` ignoriert sie
- `%` steht für beliebig viele Zeichen inkl. keinem Zeichen
- `_` steht für genau ein beliebiges Zeichen
- `'B%'` beginnt mit B, `'%B'` endet mit B, `'%B%'` enthält B
- Ohne Platzhalter muss der gesamte Text passen
- Im bisherigen Python-Code mit psycopg2 `%` verdoppeln, bspw. `'B%%'`, direkt in SQL genügt `'B%'`

### Fehlende Werte
- NULL bedeutet fehlend oder unbekannt, mit `IS NULL` oder `IS NOT NULL` prüfen
- Niemals `= NULL` oder `<> NULL` verwenden
- 0 und leerer Text `''` sind keine NULL-Werte
- Vergleiche sowie `NOT IN` und `NOT LIKE` wählen NULL nicht automatisch aus
- Sollen fehlende Werte ebenfalls erscheinen, ausdrücklich ergänzen, bspw. `price < 2000 OR price IS NULL`

### Datumsbereiche
- Datumswerte im Format `'YYYY-MM-DD'` schreiben
- Bei Zeitstempeln kann `BETWEEN '2023-01-01' AND '2023-12-31'` den letzten Tag nach Mitternacht ausschliessen
- Für das gesamte Jahr sicher filtern — `datum >= '2023-01-01' AND datum < '2024-01-01'`

### Sortieren und begrenzen
- `ORDER BY` sortiert das Ergebnis, ohne Sortierung ist keine Reihenfolge garantiert
- `ASC` aufsteigend und Standard, `DESC` absteigend
- Bei mehreren Sortierspalten entscheidet zuerst die erste, bei Gleichstand die nächste
- Die Sortierrichtung gilt je Spalte, `ORDER BY make, price DESC` bedeutet Marke aufsteigend und Preis absteigend
- `NULLS LAST` stellt fehlende Werte ans Ende, `NULLS FIRST` an den Anfang
- In PostgreSQL stehen NULL bei `ASC` standardmässig zuletzt, bei `DESC` zuerst
- `LIMIT 5` liefert maximal fünf Zeilen, `OFFSET 5` überspringt die ersten fünf
- Für verlässliche Top-N-Ausgaben und Seiten zusätzlich nach einer eindeutigen ID sortieren
- `LIMIT 5` liefert fünf Zeilen, nicht automatisch alle Datensätze mit den fünf höchsten unterschiedlichen Preisen

### Aliase und Reihenfolge
- SELECT-Aliase können in `ORDER BY` verwendet werden, in `WHERE` den ursprünglichen Spaltennamen oder Ausdruck verwenden
- Schreibreihenfolge — `SELECT → FROM → WHERE → ORDER BY → LIMIT → OFFSET`
- Merksatz: Erst filtern, dann sortieren, danach Zeilen überspringen und die Ausgabe begrenzen

## Queries

### Retrieve all rows and all columns from the bicycles table
- `SELECT *` wählt alle Spalten aus und `FROM bicycles` bestimmt die Tabelle
- Ohne `WHERE` werden alle Zeilen ausgegeben, ohne `ORDER BY` ist ihre Reihenfolge nicht festgelegt

In [4]:
# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)                                       #Engine für die Datenbankverbindung erstellen

# Execute SQL query
df_bicycles = pd.read_sql_query("""
    SELECT *                                                --Alle Spalten auswählen
    FROM bicycles;                                          --Daten aus der Tabelle bicycles abrufen
""", con=engine)                                                                #SQL ausführen und Ergebnis als DataFrame speichern

# Dispose the engine
engine.dispose()                                                                #Verbindungen im Pool schliessen und Ressourcen freigeben

# Show results
display(df_bicycles)                                                            #Ergebnistabelle anzeigen

,bicycleid,make,model,year,framenr,price,status
0,C7ZGWFLBUV,Trek,Domane,2019,A15 0LX,5066.94,In Maintenance
1,C875HZ1AG6,Merida,Reacto,2021,435N,5529.38,Sold
2,CDONEWSWFP,Merida,Big Nine,2021,QBA-572,1518.33,In Maintenance
3,C2UQZP803C,Trek,Marlin,2023,5-92778W,5038.99,In Maintenance
4,C2WFIK6WEP,Merida,Scultura,2020,300 VLT,1548.19,Sold
...,...,...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2020,911 LYI,2376.02,Available
1996,CP3F48BE9Y,Merida,Scultura,2022,YPU 135,2833.36,Available
1997,CHBXQ3SSNU,Merida,Scultura,2019,059-GNM,1629.91,In Maintenance
1998,CP5CTME2ZD,Trek,Marlin,2021,02V 6510,6176.55,In Maintenance


## Tipps
#### display(df_bicycles) bspw. in einen neuen Codeblock eingeben, um den Dataframe anzuzeigen

In [7]:
display(df_bicycles)

,bicycleid,make,model,year,framenr,price,status
0,C7ZGWFLBUV,Trek,Domane,2019,A15 0LX,5066.94,In Maintenance
1,C875HZ1AG6,Merida,Reacto,2021,435N,5529.38,Sold
2,CDONEWSWFP,Merida,Big Nine,2021,QBA-572,1518.33,In Maintenance
3,C2UQZP803C,Trek,Marlin,2023,5-92778W,5038.99,In Maintenance
4,C2WFIK6WEP,Merida,Scultura,2020,300 VLT,1548.19,Sold
...,...,...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2020,911 LYI,2376.02,Available
1996,CP3F48BE9Y,Merida,Scultura,2022,YPU 135,2833.36,Available
1997,CHBXQ3SSNU,Merida,Scultura,2019,059-GNM,1629.91,In Maintenance
1998,CP5CTME2ZD,Trek,Marlin,2021,02V 6510,6176.55,In Maintenance


### Retrieve all distinct bicycle makes from the bicycles table
- `SELECT DISTINCT make` gibt jede vorhandene Fahrradmarke nur einmal aus

In [8]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT DISTINCT make        --Jede Fahrradmarke nur einmal ausgeben
    FROM bicycles;              --Daten aus der Tabelle bicycles abrufen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make
0,Santa Cruz
1,Merida
2,Bianchi
3,Trek
4,Giant
5,Specialized
6,Cube
7,Scott
8,Cannondale
9,Cervelo


### Retrieve all distinct combinations of bicycle make and model. Sort by make and then by model in ascending order
- `DISTINCT` entfernt doppelte Kombinationen aller ausgewählten Spalten. Eine Marke kann mehrfach erscheinen, wenn sie unterschiedliche Modelle hat

In [9]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT DISTINCT make,                   --Marke und Modell gemeinsam auf eindeutige Kombinationen reduzieren
                    model                   --Dieselbe Marke mit einem anderen Modell bleibt erhalten
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    ORDER BY make ASC, model ASC;           --Zuerst nach Marke, innerhalb der Marke nach Modell sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,model
0,Bianchi,Aria
1,Bianchi,Infinito
2,Bianchi,Oltre
3,Cannondale,SuperSix
4,Cannondale,Synapse
5,Cannondale,Trail
6,Cervelo,Caledonia
7,Cervelo,R5
8,Cervelo,Soloist
9,Cube,Agree


### Retrieve the bicycle ID, make, model and price of all bicycles
- `SELECT bicycleid, make, model, price` wählt nur die vier gewünschten Spalten aus
- Ohne `WHERE` werden alle Fahrräder ausgegeben, weitere Tabellenspalten bleiben im Ergebnis ausgeblendet

In [10]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,           --Eindeutige Fahrrad-ID auswählen
           make,                --Fahrradmarke auswählen
           model,               --Fahrradmodell auswählen
           price                --Listenpreis auswählen; kein Komma vor FROM
    FROM bicycles;              --Daten aus der Tabelle bicycles abrufen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2376.02
1996,CP3F48BE9Y,Merida,Scultura,2833.36
1997,CHBXQ3SSNU,Merida,Scultura,1629.91
1998,CP5CTME2ZD,Trek,Marlin,6176.55


### Retrieve the make, model and price of all bicycles. Rename the price column to list_price in the result
- `SELECT make, model, price` wählt nur die drei gewünschten Spalten aus
- `price AS list_price` benennt die Preisspalte nur im Ergebnis um, die Originalspalte bleibt unverändert

In [11]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                --Fahrradmarke auswählen
           model,               --Fahrradmodell auswählen
           price AS list_price  --Preisspalte im Ergebnis umbenennen, Originalspalte bleibt unverändert
    FROM bicycles;              --Daten aus der Tabelle bicycles abrufen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,model,list_price
0,Trek,Domane,5066.94
1,Merida,Reacto,5529.38
2,Merida,Big Nine,1518.33
3,Trek,Marlin,5038.99
4,Merida,Scultura,1548.19
...,...,...,...
1995,Trek,Emonda,2376.02
1996,Merida,Scultura,2833.36
1997,Merida,Scultura,1629.91
1998,Trek,Marlin,6176.55


### Retrieve the bicycle ID, make, model and price of all Bianchi bicycles
- `WHERE make = 'Bianchi'` gibt nur Fahrräder der Marke Bianchi aus
- Textwerte stehen in einfachen Anführungszeichen, die Schreibweise muss exakt übereinstimmen

In [12]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,           --Fahrrad-ID auswählen
           make,                --Fahrradmarke auswählen
           model,               --Fahrradmodell auswählen
           price                --Listenpreis auswählen
    FROM bicycles               --Daten aus der Tabelle bicycles abrufen
    WHERE make = 'Bianchi';     --Nur Fahrräder der Marke Bianchi auswählen, Textwerte in einfachen Anführungszeichen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Retrieve the bicycle ID, make, model and price of all bicycles with a price greater than 3000
- `WHERE price > 3000` gibt nur Fahrräder mit einem Preis über 3000 aus, genau 3000 ist ausgeschlossen
- Zahlen stehen ohne Anführungszeichen, fehlende Preise werden durch diesen Vergleich nicht ausgewählt

In [13]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,           --Fahrrad-ID auswählen
           make,                --Fahrradmarke auswählen
           model,               --Fahrradmodell auswählen
           price                --Listenpreis auswählen
    FROM bicycles               --Daten aus der Tabelle bicycles abrufen
    WHERE price > 3000;         --Nur Preise über 3000 auswählen; Zahlen ohne Anführungszeichen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,C2UQZP803C,Trek,Marlin,5038.99
3,C4RQCO25I9,Cube,Reaction,3339.27
4,C9LPTOS9KE,Cannondale,Synapse,5399.61
...,...,...,...,...
1389,CFFTG1MNW0,Bianchi,Aria,5939.92
1390,C6DHR3STN7,Trek,Marlin,6624.90
1391,CN5W80Q44T,Santa Cruz,Bronson,5442.78
1392,CX9QJEJHDE,Giant,Talon,5264.70


### Retrieve the bicycle ID, make, model and price of all bicycles priced from 2000 to 4000 inclusive, excluding bicycles made by Bianchi
- Vergleichsoperatoren: `=` gleich, `<>` ungleich, `<` kleiner, `<=` kleiner oder gleich, `>` grösser, `>=` grösser oder gleich

In [14]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price >= 2000                     --Untere Grenze einschliessen: mindestens 2000
      AND price <= 4000                     --Obere Grenze einschliessen: höchstens 4000
      AND make <> 'Bianchi';                --Bianchi ausschliessen; NULL-Marken werden ebenfalls nicht ausgewählt
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C4RQCO25I9,Cube,Reaction,3339.27
1,CWGKVNNCYD,Giant,TCR,2087.55
2,CCI58KF4MJ,Cube,Stereo,3047.13
3,CNKHIRR7HH,Santa Cruz,Bronson,2314.02
4,C4ZC4AYR1E,Trek,Emonda,2198.88
...,...,...,...,...
479,CK9BRGCOBR,Merida,Reacto,2501.10
480,CXSPK20YP0,Cervelo,R5,3719.35
481,CDBAWQPE4N,Cervelo,Soloist,2448.92
482,CLH919EIKV,Trek,Emonda,2376.02


### Retrieve the bicycle ID, make, model and price of all Bianchi bicycles with a price greater than 3000
- Vergleichsoperatoren: `=` gleich, `<>` ungleich, `<` kleiner, `<=` kleiner oder gleich, `>` grösser, `>=` grösser oder gleich

In [15]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,           --Fahrrad-ID auswählen
           make,                --Fahrradmarke auswählen
           model,               --Fahrradmodell auswählen
           price                --Listenpreis auswählen
    FROM bicycles               --Daten aus der Tabelle bicycles abrufen
    WHERE make = 'Bianchi'      --Erste Bedingung: Marke ist Bianchi (in Anführungszeichen)
      AND price > 3000;         --Zweite Bedingung: Preis über 3000; beide Bedingungen müssen erfüllt sein
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CA2FL7YUGR,Bianchi,Aria,6053.73
...,...,...,...,...
139,C46AFS3V3O,Bianchi,Aria,5965.55
140,C6FYCWDJKF,Bianchi,Infinito,6983.40
141,CVG2ERXCAM,Bianchi,Oltre,3068.96
142,CB4PK8PLSN,Bianchi,Oltre,7765.04


### Retrieve the bicycle ID, make, model and price of all bicycles that are either made by Bianchi or have a price below 1000
- `OR` wählt Fahrräder aus, die mind. eine Bedingung erfüllen: Marke Bianchi oder Preis unter 1000
- Auch Fahrräder, die beide Bedingungen erfüllen, werden ausgegeben und erscheinen dabei nur einmal pro Tabellenzeile

In [16]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,           --Fahrrad-ID auswählen
           make,                --Fahrradmarke auswählen
           model,               --Fahrradmodell auswählen
           price                --Listenpreis auswählen
    FROM bicycles               --Daten aus der Tabelle bicycles abrufen
    WHERE make = 'Bianchi'      --Erste Bedingung: Marke ist Bianchi
       OR price < 1000;         --Zweite Bedingung: Preis unter 1000; mindestens eine Bedingung muss erfüllt sein
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CCKYJCLZKE,Santa Cruz,Bronson,802.81
2,CJ3K49UQS0,Giant,TCR,918.55
3,CTFUCPJ6CQ,Bianchi,Aria,7050.68
4,CCP6K2ZAAU,Bianchi,Aria,3965.39
...,...,...,...,...
229,C49VNDE7VN,Giant,Defy,916.97
230,CLKN05TSC9,Specialized,Rockhopper,878.78
231,CB4PK8PLSN,Bianchi,Oltre,7765.04
232,CFFTG1MNW0,Bianchi,Aria,5939.92


### Retrieve the bicycle ID, make, model and price of all Bianchi or Trek bicycles with a price greater than 3000
- `AND` bindet stärker als `OR`. Mit Klammern wird festgelegt, dass die Preisbedingung für beide Marken gilt

In [17]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                            --Fahrrad-ID auswählen
           make,                                 --Fahrradmarke auswählen
           model,                                --Fahrradmodell auswählen
           price                                 --Listenpreis auswählen
    FROM bicycles                                --Daten aus der Tabelle bicycles abrufen
    WHERE (make = 'Bianchi' OR make = 'Trek')    --Zuerst beide erlaubten Marken zusammenfassen
      AND price > 3000;                          --Preisbedingung muss für beide Marken erfüllt sein
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C2UQZP803C,Trek,Marlin,5038.99
2,C2ZEK0YPD9,Bianchi,Oltre,7382.81
3,CE5XORIKH7,Trek,Emonda,7600.96
4,CB46L6E7V3,Trek,Emonda,4396.75
...,...,...,...,...
267,CB4PK8PLSN,Bianchi,Oltre,7765.04
268,C1V12YL0CG,Trek,Domane,7497.23
269,CFFTG1MNW0,Bianchi,Aria,5939.92
270,C6DHR3STN7,Trek,Marlin,6624.90


### Retrieve the bicycle ID, make, model and price of all bicycles that are not made by Bianchi
- `WHERE NOT make = 'Bianchi'` schliesst Fahrräder der Marke Bianchi aus, gleichbedeutend mit `WHERE make <> 'Bianchi'`
- Fehlende Marken werden ebenfalls nicht ausgewählt, dafür wäre zusätzlich `OR make IS NULL` nötig

In [18]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,               --Fahrrad-ID auswählen
           make,                    --Fahrradmarke auswählen
           model,                   --Fahrradmodell auswählen
           price                    --Listenpreis auswählen
    FROM bicycles                   --Daten aus der Tabelle bicycles abrufen
    WHERE NOT make = 'Bianchi';     --Bianchi ausschliessen; entspricht make <> 'Bianchi'; NULL wird nicht ausgewählt
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1804,CX9QJEJHDE,Giant,Talon,5264.70
1805,CLH919EIKV,Trek,Emonda,2376.02
1806,CP3F48BE9Y,Merida,Scultura,2833.36
1807,CHBXQ3SSNU,Merida,Scultura,1629.91


### Retrieve the bicycle ID, make, model and price of all bicycles with a price between 2000 and 4000, including both limits
- `WHERE price BETWEEN 2000 AND 4000` wählt Preise von 2000 bis 4000 inkl. beider Grenzen aus
- Entspricht `WHERE price >= 2000 AND price <= 4000`, fehlende Preise werden nicht ausgewählt

In [19]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price BETWEEN 2000 AND 4000;      --Preise von 2000 bis 4000 auswählen; beide Grenzen sind eingeschlossen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C4RQCO25I9,Cube,Reaction,3339.27
1,CWGKVNNCYD,Giant,TCR,2087.55
2,CCI58KF4MJ,Cube,Stereo,3047.13
3,CNKHIRR7HH,Santa Cruz,Bronson,2314.02
4,C4ZC4AYR1E,Trek,Emonda,2198.88
...,...,...,...,...
536,CXSPK20YP0,Cervelo,R5,3719.35
537,CDBAWQPE4N,Cervelo,Soloist,2448.92
538,CLH919EIKV,Trek,Emonda,2376.02
539,CP3F48BE9Y,Merida,Scultura,2833.36


### Retrieve the bicycle ID, make, model and price of all bicycles with a price outside the range of 2000 to 4000
- `WHERE price NOT BETWEEN 2000 AND 4000` wählt Preise unter 2000 oder über 4000 aus, die beiden Grenzen sind ausgeschlossen
- Entspricht `WHERE price < 2000 OR price > 4000`, fehlende Preise werden nicht ausgewählt

In [20]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                           --Fahrrad-ID auswählen
           make,                                --Fahrradmarke auswählen
           model,                               --Fahrradmodell auswählen
           price                                --Listenpreis auswählen
    FROM bicycles                               --Daten aus der Tabelle bicycles abrufen
    WHERE price NOT BETWEEN 2000 AND 4000;      --Preise unter 2000 oder über 4000 auswählen; Grenzen und NULL ausgeschlossen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1454,C6DHR3STN7,Trek,Marlin,6624.90
1455,CN5W80Q44T,Santa Cruz,Bronson,5442.78
1456,CX9QJEJHDE,Giant,Talon,5264.70
1457,CHBXQ3SSNU,Merida,Scultura,1629.91


### Retrieve the sale ID, bicycle ID, sale date and sale price of all sales from March to December 2023 inclusive. Sort by sale date in ascending order
- Datumswerte in einfachen Anführungszeichen schreiben: `'YYYY-MM-DD'`
- Die untere Grenze wird eingeschlossen, die obere ausgeschlossen. Dadurch ist der gesamte Dezember enthalten, auch wenn die Datumswerte Uhrzeiten enthalten

In [21]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT saleid,                            --Verkaufs-ID auswählen
           bicycleid,                         --ID des verkauften Fahrrads auswählen
           saledate,                          --Verkaufsdatum auswählen
           saleprice                          --Tatsächlichen Verkaufspreis auswählen
    FROM sales                                --Verkäufe stehen in der Tabelle sales
    WHERE saledate >= '2023-03-01'            --Ab dem 1. März 2023 einschliesslich
      AND saledate < '2024-01-01'             --Vor dem 1. Januar 2024, gesamter Dezember enthalten
    ORDER BY saledate ASC, saleid ASC;        --Chronologisch sortieren, bei gleichem Datum nach Verkaufs-ID
""", con=engine)

engine.dispose()
display(df_spickap01)

,saleid,bicycleid,saledate,saleprice
0,S8L5Z53WA0,C7Q6H9R9HU,2023-03-04,4650
1,S22JXD2CQQ,CEJ6O76MUA,2023-03-09,3480
2,SKSXSCWFKA,C2GGF5IKFQ,2023-03-11,4640
3,SF3VJK1X3C,CB8P29T8HN,2023-03-13,4560
4,S5OW4GRZJ7,CFGVCBYLYK,2023-03-29,2190
...,...,...,...,...
116,STGZ9BS52X,CLQCGBXCAG,2023-12-25,2830
117,S5FKM5VRRG,CS75T5RDH7,2023-12-26,2110
118,S9WJAHWEU2,CRS8LAJ1MN,2023-12-27,3460
119,S75D9BOPLT,C3RHPOYHGV,2023-12-31,3440


### Retrieve the bicycle ID, make, model and price of all bicycles with a price of either 2087.55, 2943.58 or 2448.92
- `WHERE price IN (2087.55, 2943.58, 2448.92)` wählt Fahrräder aus, deren Preis exakt einem der drei Werte entspricht
- Entspricht drei mit `OR` verbundenen Preisvergleichen, die Werte bilden keinen Bereich

In [22]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                                --Fahrrad-ID auswählen
           make,                                     --Fahrradmarke auswählen
           model,                                    --Fahrradmodell auswählen
           price                                     --Listenpreis auswählen
    FROM bicycles                                    --Daten aus der Tabelle bicycles abrufen
    WHERE price IN (2087.55, 2943.58, 2448.92);      --Nur diese drei exakten Preise auswählen, entspricht drei mit OR verbundenen Vergleichen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,CWGKVNNCYD,Giant,TCR,2087.55
1,CDBAWQPE4N,Cervelo,Soloist,2448.92
2,CNJYADLN8R,Bianchi,Aria,2943.58


### Retrieve the bicycle ID, make, model and price of all bicycles whose price is not 2000, 3000 or 4000
- `WHERE price NOT IN (2000, 3000, 4000)` schliesst genau diese drei Preise aus, andere Preise bleiben enthalten
- Fehlende Preise werden ebenfalls nicht ausgewählt, dafür wäre zusätzlich `OR price IS NULL` nötig

In [23]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                           --Fahrrad-ID auswählen
           make,                                --Fahrradmarke auswählen
           model,                               --Fahrradmodell auswählen
           price                                --Listenpreis auswählen
    FROM bicycles                               --Daten aus der Tabelle bicycles abrufen
    WHERE price NOT IN (2000, 3000, 4000);      --Diese drei exakten Preise ausschliessen; NULL-Preise werden ebenfalls nicht ausgewählt
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2376.02
1996,CP3F48BE9Y,Merida,Scultura,2833.36
1997,CHBXQ3SSNU,Merida,Scultura,1629.91
1998,CP5CTME2ZD,Trek,Marlin,6176.55


### Retrieve the bicycle ID, make, model and price of all bicycles whose make starts with an uppercase B
- `LIKE 'B%'` sucht Marken mit grossem B am Anfang, `%` steht für beliebig viele Zeichen inkl. keinem Zeichen
- Varianten — `LIKE 'b%'` nur kleines b, `ILIKE 'b%'` grosses oder kleines B, `LIKE '%B'` endet mit B, `LIKE '%B%'` enthält B, `LIKE 'B_'` B gefolgt von genau einem Zeichen
- Im bisherigen Python-Code mit psycopg2 `%` verdoppeln, bspw. `LIKE 'B%%'`, direkt in SQL genügt `LIKE 'B%'`

In [24]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE make LIKE 'B%%';                  --Beginnt mit B; Gross-/Kleinschreibung beachten; Platzhalter für psycopg2 verdoppeln
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Retrieve the bicycle ID, make, model and price of all bicycles whose make starts with B or b, regardless of case
- `ILIKE 'b%'` sucht Marken mit B oder b am Anfang, `ILIKE` ignoriert Gross- und Kleinschreibung
- Varianten: `ILIKE '%b'` endet mit B oder b, `ILIKE '%b%'` enthält B oder b, `ILIKE 'b_'` B oder b gefolgt von genau einem Zeichen, `ILIKE 'b'` nur B oder b, `NOT ILIKE 'b%'` beginnt weder mit B noch mit b
- Im bisherigen Python-Code mit psycopg2 `%` verdoppeln, bspw. `ILIKE '%%b%%'`, direkt in SQL genügt `ILIKE '%b%'`

In [25]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE make ILIKE 'b%%';                 --ILIKE ignoriert Gross-/Kleinschreibung; Platzhalter für psycopg2 verdoppeln
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Retrieve the bicycle ID, make, model and price of all bicycles whose make contains the text "ian", regardless of case
- `ILIKE '%ian%'` sucht den Text ian an beliebiger Stelle, unabhängig von Gross- und Kleinschreibung
- Varianten: `ILIKE 'ian%'` beginnt mit ian, `ILIKE '%ian'` endet mit ian, `ILIKE 'ian'` entspricht exakt ian, `NOT ILIKE '%ian%'` enthält kein ian, `LIKE '%ian%'` enthält ian mit exakt dieser Schreibweise
- Im bisherigen Python-Code mit psycopg2 `%` verdoppeln, bspw. `ILIKE '%%ian%%'`

In [26]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE make ILIKE '%%ian%%';             --Text ian an beliebiger Stelle suchen; Gross-/Kleinschreibung ignorieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CWGKVNNCYD,Giant,TCR,2087.55
2,CJ3K49UQS0,Giant,TCR,918.55
3,CTFUCPJ6CQ,Bianchi,Aria,7050.68
4,CP64NZ2VJ0,Giant,Talon,4219.20
...,...,...,...,...
404,CB4PK8PLSN,Bianchi,Oltre,7765.04
405,CULZ43GQ7W,Giant,Defy,1759.41
406,CFFTG1MNW0,Bianchi,Aria,5939.92
407,CX9QJEJHDE,Giant,Talon,5264.70


### Retrieve the bicycle ID, make, model and price of all bicycles whose make ends with "chi", regardless of case
- `ILIKE '%chi'` sucht Marken mit chi am Ende, unabhängig von Gross- und Kleinschreibung
- Varianten: `ILIKE 'chi%'` beginnt mit chi, `ILIKE '%chi%'` enthält chi, `ILIKE 'chi'` entspricht exakt chi, `NOT ILIKE '%chi'` endet nicht mit chi, `LIKE '%chi'` endet mit exakt kleingeschriebenem chi
- Im bisherigen Python-Code mit psycopg2 `%` verdoppeln, bspw. `ILIKE '%%chi'`

In [27]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE make ILIKE '%%chi';               --Marke endet mit chi; davor beliebig viele Zeichen; Gross-/Kleinschreibung ignorieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Retrieve the bicycle ID, make, model and price of all bicycles whose make does not start with B or b
- `NOT ILIKE 'b%'` schliesst Marken mit B oder b am Anfang aus, fehlende Marken werden ebenfalls nicht ausgewählt
- Varianten: `NOT ILIKE '%b'` endet nicht mit B oder b, `NOT ILIKE '%b%'` enthält weder B noch b, `NOT ILIKE 'b'` ist nicht exakt B oder b, `NOT LIKE 'B%'` schliesst nur Marken mit grossem B am Anfang aus
- Im bisherigen Python-Code mit psycopg2 `%` verdoppeln, bspw. `NOT ILIKE 'b%%'`

In [28]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE make NOT ILIKE 'b%%';             --Marken mit B oder b am Anfang ausschliessen; NULL wird ebenfalls nicht ausgewählt
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1804,CX9QJEJHDE,Giant,Talon,5264.70
1805,CLH919EIKV,Trek,Emonda,2376.02
1806,CP3F48BE9Y,Merida,Scultura,2833.36
1807,CHBXQ3SSNU,Merida,Scultura,1629.91


### Retrieve the bicycle ID, make, model and price of all bicycles whose make has exactly seven characters and starts with an uppercase B
- `LIKE 'B______'` sucht Marken mit grossem B gefolgt von genau sechs Zeichen, insgesamt also sieben Zeichen
- Varianten: `LIKE 'b______'` beginnt mit kleinem b, `ILIKE 'b______'` beginnt mit B oder b, `LIKE '______B'` endet mit grossem B bei insgesamt sieben Zeichen
- Jeder `_` steht für genau ein beliebiges Zeichen, `%` dagegen für beliebig viele Zeichen inkl. keinem Zeichen

In [29]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE make LIKE 'B______';              --B gefolgt von genau sechs Zeichen; jeder Unterstrich steht für genau ein Zeichen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Retrieve the bicycle ID, make, model and price of all bicycles with a missing price
- `NULL` bedeutet fehlender oder unbekannter Wert, ein Preis von 0 ist kein NULL

In [30]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price IS NULL;                    --Fehlende Preise auswählen; IS NULL verwenden, niemals = NULL
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price


### Retrieve the bicycle ID, make, model and price of all bicycles with a known price
- Ein Preis von 0 bleibt enthalten, `IS NOT NULL` prüft nur, ob ein Wert vorhanden ist

In [31]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price IS NOT NULL;                --Nur vorhandene Preise auswählen; ein Preis von 0 bleibt enthalten
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2376.02
1996,CP3F48BE9Y,Merida,Scultura,2833.36
1997,CHBXQ3SSNU,Merida,Scultura,1629.91
1998,CP5CTME2ZD,Trek,Marlin,6176.55


### Retrieve the bicycle ID, make, model and price of all bicycles. Sort the result by price in ascending order, with missing prices last
- `ORDER BY price ASC` sortiert vom günstigsten zum teuersten Preis, `DESC` umgekehrt
- `NULLS LAST` stellt fehlende Preise ans Ende

In [32]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    ORDER BY price ASC NULLS LAST;          --Nach Preis aufsteigend sortieren. Fehlende Preise zuletzt anzeigen, alternativ "DESC"
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,CS2RTMPG37,Specialized,Allez,800.26
1,CCKYJCLZKE,Santa Cruz,Bronson,802.81
2,CZVM7LWXRP,Specialized,Rockhopper,809.00
3,CNX3HEE5GS,Merida,Big Nine,813.47
4,C7ZPXA6EXM,Cannondale,SuperSix,822.42
...,...,...,...,...
1995,CTVKH7S3PZ,Trek,Marlin,7972.58
1996,CLQCGBXCAG,Merida,Scultura,7972.88
1997,CJSLUUOM7Z,Santa Cruz,Hightower,7984.31
1998,CENB0TLQKE,Bianchi,Oltre,7989.74


### Retrieve the bicycle ID, make, model and price of all bicycles. Sort by make in ascending order and, within each make, by price in descending order, with missing prices last
- `ORDER BY make ASC, price DESC` sortiert Marken alphabetisch und innerhalb jeder Marke die teuersten Fahrräder zuerst
- `NULLS LAST` hinter `price DESC` stellt fehlende Preise innerhalb jeder Marke ans Ende

In [33]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    ORDER BY make ASC,                      --Zuerst nach Marke alphabetisch aufsteigend sortieren
             price DESC NULLS LAST;         --Innerhalb derselben Marke nach Preis absteigend; fehlende Preise zuletzt
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,CENB0TLQKE,Bianchi,Oltre,7989.74
1,C7JUNJQSTR,Bianchi,Infinito,7963.11
2,C3MZ4T3WRF,Bianchi,Oltre,7951.93
3,CVSWF385E4,Bianchi,Oltre,7859.94
4,CV8A3G1LOS,Bianchi,Oltre,7824.00
...,...,...,...,...
1995,C1U71ESAXN,Trek,Emonda,1109.75
1996,CYEUN8XEM0,Trek,Domane,1091.15
1997,C32MK7YOH3,Trek,Domane,1074.92
1998,CYQC3N4R7I,Trek,Marlin,1054.77


### Retrieve the bicycle ID, make, model and price of the five most expensive bicycles with a known price. If prices are equal, sort by bicycle ID in ascending order
- `WHERE price IS NOT NULL` schliesst fehlende Preise aus, `ORDER BY price DESC, bicycleid ASC` sortiert teuerste zuerst und bei Preisgleichheit nach ID
- `LIMIT 5` gibt nur die ersten fünf Zeilen des sortierten Ergebnisses aus

In [34]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price IS NOT NULL                 --Fahrräder mit fehlendem Preis ausschliessen
    ORDER BY price DESC,                    --Teuerste Fahrräder zuerst anzeigen
             bicycleid ASC                  --Bei gleichem Preis nach Fahrrad-ID aufsteigend sortieren
    LIMIT 5;                                --Ausgabe auf maximal fünf Zeilen begrenzen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C187CGZK39,Trek,Marlin,7995.90
1,CENB0TLQKE,Bianchi,Oltre,7989.74
2,CJSLUUOM7Z,Santa Cruz,Hightower,7984.31
3,CLQCGBXCAG,Merida,Scultura,7972.88
4,CTVKH7S3PZ,Trek,Marlin,7972.58


### Retrieve the bicycle ID, make, model and price of the sixth to tenth most expensive bicycles with a known price. If prices are equal, sort by bicycle ID in ascending order
- `WHERE price IS NOT NULL` schliesst fehlende Preise aus, `ORDER BY price DESC, bicycleid ASC` sortiert teuerste zuerst und bei Preisgleichheit nach ID
- `OFFSET 5` überspringt die ersten fünf Zeilen, `LIMIT 5` gibt die nächsten fünf aus, also Position 6 bis 10

In [35]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                       --Fahrrad-ID auswählen
           make,                            --Fahrradmarke auswählen
           model,                           --Fahrradmodell auswählen
           price                            --Listenpreis auswählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price IS NOT NULL                 --Fahrräder mit fehlendem Preis ausschliessen
    ORDER BY price DESC,                    --Teuerste Fahrräder zuerst anzeigen
             bicycleid ASC                  --Bei gleichem Preis nach Fahrrad-ID aufsteigend sortieren
    LIMIT 5                                 --Maximal fünf Zeilen ausgeben
    OFFSET 5;                               --Die ersten fünf Zeilen der sortierten Ergebnismenge überspringen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,CNS05492DI,Cube,Reaction,7969.30
1,CV56P1JD32,Scott,Addict,7968.66
2,CU0L6RJUXR,Specialized,Rockhopper,7964.52
3,C7JUNJQSTR,Bianchi,Infinito,7963.11
4,C1YLDNRGD4,Santa Cruz,Hightower,7962.93
